## Step 1: Import Required Libraries

We import essential libraries for data manipulation (pandas), cloud interaction (google.cloud.bigquery), and environment management (dotenv). This setup ensures we have the necessary tools for data engineering tasks in a professional environment.

In [1]:
import os
import pandas as pd
from google.cloud import bigquery
from dotenv import load_dotenv

## Step 2: Environment Configuration

We load environment variables from a .env file for security, set the BigQuery project ID, dataset, and location. This professional setup ensures secure and region-optimized access to cloud resources.

In [2]:
# 1. Professional Environment Configuration
load_dotenv()
project_id = 'mlopsmarketingproject'
dataset_id = 'olist_marketing_qualified_leads_dataset'
location = 'northamerica-northeast1'

client = bigquery.Client(project=project_id)

## Step 3: Master Query Construction (Pushdown Logic)

We construct a SQL query that performs data aggregation and joins directly in BigQuery (pushdown logic) to minimize data transfer and leverage cloud computing power. This includes calculating seller performance metrics and joining with lead data for LTV analysis.

In [3]:
# 2. Master Query (Pushdown Logic)
# We join leads with their closures and total sales (LTV)
sql_abt = f"""
WITH seller_performance AS (
    SELECT 
        seller_id, 
        SUM(price) as total_revenue,
        COUNT(order_id) as total_orders
    FROM `{project_id}.{dataset_id}.olist_order_items_dataset`
    GROUP BY 1
)
SELECT 
    mql.mql_id,
    mql.first_contact_date,
    mql.landing_page_id,
    mql.origin,
    cd.business_segment,
    cd.lead_type,
    cd.lead_behaviour_profile, -- Key profiles: cat, wolf, shark, eagle [5]
    cd.won_date,
    IF(cd.won_date IS NOT NULL, 1, 0) as converted,
    COALESCE(sp.total_revenue, 0) as ltv_revenue,
    COALESCE(sp.total_orders, 0) as orders_count
FROM `{project_id}.{dataset_id}.olist_marketing_qualified_leads_dataset` mql
LEFT JOIN `{project_id}.{dataset_id}.olist_closed_deals_dataset` cd 
    ON mql.mql_id = cd.mql_id
LEFT JOIN seller_performance sp 
    ON cd.seller_id = sp.seller_id
"""

## Step 4: Execute Query and Initial Validation

We execute the query in BigQuery and download the results as a DataFrame. Immediate validation checks the data dimensions and structure to ensure data quality before further processing.

In [4]:
print("⏳ Generating ABT in BigQuery and downloading for ML...")
df_abt = client.query(sql_abt, location=location).to_dataframe()

# 3. Immediate Quality Validation
print(f"✅ ABT Generated. Dimensions: {df_abt.shape}")
print(df_abt.info())

⏳ Generating ABT in BigQuery and downloading for ML...


c:\Users\User\Desktop\Software y Clases\BigData\OList\olist-project-sa\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


✅ ABT Generated. Dimensions: (8000, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype              
---  ------                  --------------  -----              
 0   mql_id                  8000 non-null   object             
 1   first_contact_date      8000 non-null   dbdate             
 2   landing_page_id         8000 non-null   object             
 3   origin                  7940 non-null   object             
 4   business_segment        841 non-null    object             
 5   lead_type               836 non-null    object             
 6   lead_behaviour_profile  665 non-null    object             
 7   won_date                842 non-null    datetime64[us, UTC]
 8   converted               8000 non-null   Int64              
 9   ltv_revenue             8000 non-null   float64            
 10  orders_count            8000 non-null   Int64              
dtypes: 

## Step 5: Data Processing and Imputation

We create a binary conversion flag and impute missing values in critical qualification columns with a business-relevant value ('not_qualified'). This ensures the data is clean and ready for machine learning models.

In [5]:
df_abt['converted'] = df_abt['won_date'].notna().astype(int)

# Critical qualification columns
qualification_cols = ['lead_type', 'lead_behaviour_profile', 'business_segment']

# We impute with a business-sense value
df_abt[qualification_cols] = df_abt[qualification_cols].fillna('not_qualified')

## Step 6: Data Preview

We display the first few rows of the processed DataFrame to visually inspect the data structure and ensure all transformations were applied correctly.

In [6]:
print(display(df_abt.head()))

,mql_id,first_contact_date,landing_page_id,origin,business_segment,lead_type,lead_behaviour_profile,won_date,converted,ltv_revenue,orders_count
0,6710565534bb35faf7fad5971cf885b3,2018-02-02,0c83f57c786a0b4a39efab23731c7ebc,None,not_qualified,not_qualified,not_qualified,NaT,0,0.0,0
1,4da6c4d07033d355453ce49273d591c6,2018-05-18,0c83f57c786a0b4a39efab23731c7ebc,None,not_qualified,not_qualified,not_qualified,NaT,0,0.0,0
2,d788c46ad6a6c020b5062c1a99f55b2c,2018-05-15,0c83f57c786a0b4a39efab23731c7ebc,None,not_qualified,not_qualified,not_qualified,NaT,0,0.0,0
3,9e4903ea29841bc16eab71500292e9b0,2018-05-21,0c83f57c786a0b4a39efab23731c7ebc,None,not_qualified,not_qualified,not_qualified,NaT,0,0.0,0
4,e8432fb72c61c9066957124e5a420a05,2017-10-09,1722481ac9e5371e5099dea226b5421d,None,not_qualified,not_qualified,not_qualified,NaT,0,0.0,0


None


## Step 7: Persist Data in BigQuery

We save the Analytical Base Table (ABT) back to BigQuery as a new table. This allows for easy access by other team members and enables further cloud-based analytics or ML pipelines.

In [7]:
# We save the ABT as a new table in BigQuery
destination_table = f"{project_id}.{dataset_id}.abt_marketing_leads"

df_abt.to_gbq(
    destination_table=destination_table,
    project_id=project_id,
    if_exists='replace', # Or 'append' if it were a daily process
    location=location
)
print(f"✅ ABT persisted in BigQuery: {destination_table}")

C:\Users\User\AppData\Local\Temp\ipykernel_18144\3616928861.py:4: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  df_abt.to_gbq(


✅ ABT persisted in BigQuery: mlopsmarketingproject.olist_marketing_qualified_leads_dataset.abt_marketing_leads


**Note**: For cost-optimization during model tuning, a local Parquet version is also maintained to reduce cloud API calls

## Step 8: Local Backup in Parquet Format

For cost optimization during model development and to reduce cloud API calls, we also save a local copy in Parquet format. Parquet preserves data types and compresses the file size efficiently.

In [8]:
# Parquet preserves types (datetimes, ints, floats) and compresses size
df_abt.to_parquet('C:\\Users\\User\\Desktop\\Software y Clases\\BigData\\OList\\olist-project-sa\\Data\\Processed\\abt_marketing.parquet', index=False)